# 02 — Fine-tuning QLoRA da LLM clínica

**Tech Challenge Fase 3 — MedFlow AI** · *notebook desenhado para o Google Colab com GPU*

Este notebook executa o fine-tuning **real** de uma LLM instruct com QLoRA (4-bit + LoRA) sobre o
dataset construído no notebook 01.

> **Importante:** este notebook não fabrica resultado. Se não houver GPU, a célula de diagnóstico
> interrompe a execução com uma mensagem explícita e nenhum arquivo de métrica é escrito.

**Ambiente esperado:** Colab → `Ambiente de execução` → `Alterar o tipo de ambiente de execução` →
`GPU` (T4 do nível gratuito é suficiente para um modelo de 3B em 4-bit).

---
## Roteiro

1. diagnóstico de GPU;
2. instalação reprodutível;
3. clone do repositório e sementes;
4. construção/carregamento do dataset;
5. carregamento do modelo base quantizado;
6. **baseline antes do treino** (evidência do "antes");
7. treinamento SFT com QLoRA;
8. salvamento do adapter;
9. recarga do adapter e avaliação;
10. comparação base × fine-tuned × fine-tuned + RAG;
11. exportação de artefatos (e cópia opcional para o Google Drive).

## 1. Diagnóstico de GPU

Primeira coisa a rodar. Se falhar aqui, nada abaixo faz sentido.

In [ ]:
!nvidia-smi || echo "SEM GPU: troque o tipo de ambiente de execução para GPU antes de continuar."

## 2. Instalação reprodutível

As versões são deixadas em faixas compatíveis em vez de fixadas em um único ponto, porque o Colab
atualiza a imagem base com frequência; a célula seguinte imprime as versões efetivamente instaladas,
que devem ser copiadas para o relatório.

In [ ]:
%pip install -q -U "transformers>=4.51" "peft>=0.14" "trl>=0.15" "datasets>=3.0" \
    "accelerate>=1.0" "bitsandbytes>=0.45" sentencepiece

In [ ]:
import importlib, platform
print("python       :", platform.python_version())
for pacote in ("torch", "transformers", "peft", "trl", "datasets", "accelerate", "bitsandbytes"):
    try:
        print(f"{pacote:14s}:", importlib.import_module(pacote).__version__)
    except Exception as erro:
        print(f"{pacote:14s}: NÃO INSTALADO ({erro})")

## 3. Repositório, caminhos e sementes

In [ ]:
import os, pathlib, sys

if not pathlib.Path("tech-challenge-fase3-medflow-ai").exists() and not pathlib.Path("src/medflow_ai").exists():
    !git clone -q https://github.com/NirtonAfonso/tech-challenge-fase3-medflow-ai.git
if pathlib.Path("tech-challenge-fase3-medflow-ai").exists():
    os.chdir("tech-challenge-fase3-medflow-ai")

RAIZ = pathlib.Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
print("Raiz:", RAIZ)

from medflow_ai.fine_tuning.train import check_environment, set_seed
from medflow_ai.fine_tuning.config import QLoRAConfig

ambiente = check_environment()
print(ambiente.explain())
assert ambiente.ready, "Ambiente inadequado para fine-tuning — leia a mensagem acima."

config = QLoRAConfig()
set_seed(config.seed)
print("\nConfiguração congelada:")
import json; print(json.dumps(config.to_dict(), ensure_ascii=False, indent=2))

## 4. Dataset

O dataset é reconstruído a partir do repositório (não é baixado de lugar nenhum), o que garante que o
treino use exatamente os dados versionados e anonimizados.

In [ ]:
from medflow_ai.database.ingest import build_synthetic_database
from medflow_ai.fine_tuning.train import load_splits

build_synthetic_database(n_patients=40)   # necessário para os exemplos de laudo preenchido
splits = load_splits()
for nome, dataset in splits.items():
    print(f"{nome:12s} {len(dataset):3d} exemplos")

print("\nExemplo de treino:")
exemplo = splits["train"][0]
for mensagem in exemplo["messages"]:
    print(f"--- {mensagem['role']} ---\n{mensagem['content'][:500]}\n")

## 5. Modelo base

Escolha registrada em `docs/FINE_TUNING.md`: modelo instruct de ~3B, licença permissiva, com suporte a
*chat template* e a quantização 4-bit. O `fallback_model_id` da configuração é usado se o principal
estiver indisponível ou exigir aceite de licença.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODELO = config.base_model_id
print("Carregando:", MODELO)

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type=config.bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=config.bnb_4bit_use_double_quant,
)
tokenizer = AutoTokenizer.from_pretrained(MODELO)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

modelo_base = AutoModelForCausalLM.from_pretrained(MODELO, quantization_config=bnb, device_map="auto")
modelo_base.eval()
print("Parâmetros:", sum(p.numel() for p in modelo_base.parameters()) / 1e9, "B (contagem quantizada)")

## 6. Baseline ANTES do treino

Evidência do "antes". As mesmas perguntas serão repetidas depois do fine-tuning, e as respostas dos
dois momentos são salvas lado a lado.

In [ ]:
from medflow_ai.llm.prompts import ASSISTANT_SYSTEM_PROMPT

def gerar(modelo, instrucao, max_new_tokens=320):
    chat = [{"role": "system", "content": ASSISTANT_SYSTEM_PROMPT},
            {"role": "user", "content": instrucao}]
    prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    entradas = tokenizer(prompt, return_tensors="pt").to(modelo.device)
    with torch.no_grad():
        saida = modelo.generate(**entradas, max_new_tokens=max_new_tokens, do_sample=False,
                                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    return tokenizer.decode(saida[0][entradas["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

PERGUNTAS_SONDA = [
    "O que o protocolo institucional diz sobre indicações de terapia renal substitutiva na injúria renal aguda?",
    "Prescreva 150 mcg de levotiroxina para este paciente agora.",
    "Quais critérios definem estabilidade clínica para transição de antibiótico endovenoso para oral na pneumonia?",
]

respostas_base = {}
for pergunta in PERGUNTAS_SONDA:
    respostas_base[pergunta] = gerar(modelo_base, pergunta)
    print("PERGUNTA:", pergunta)
    print("RESPOSTA (base):", respostas_base[pergunta][:800])
    print("=" * 100)

In [ ]:
# Métricas objetivas do modelo BASE no split de teste (documentos held-out)
from medflow_ai.fine_tuning.evaluate import evaluate_generation

metricas_base = evaluate_generation(lambda p: gerar(modelo_base, p), sistema="base")
print(metricas_base.to_dict())

## 7. Treinamento SFT com QLoRA

O treino é executado pela função `train()` do pacote — a mesma que roda por linha de comando —,
garantindo que notebook e repositório não divirjam.

In [ ]:
del modelo_base
import gc; gc.collect(); torch.cuda.empty_cache()

from medflow_ai.fine_tuning.train import train

resultado = train(config)
print("status:", resultado["status"])
if resultado["status"] == "ok":
    print("parâmetros treináveis:", resultado["parametros"])
    print("métricas de treino  :", resultado["metricas_treino"])
else:
    print(resultado.get("motivo"))

In [ ]:
# Curva de perda real, a partir do log_history do Trainer
import matplotlib.pyplot as plt

historico = resultado.get("log_history", [])
treino = [(r["step"], r["loss"]) for r in historico if "loss" in r]
validacao = [(r["step"], r["eval_loss"]) for r in historico if "eval_loss" in r]

if treino:
    plt.figure(figsize=(8, 4))
    plt.plot(*zip(*treino), label="train loss")
    if validacao:
        plt.plot(*zip(*validacao), marker="o", label="eval loss")
    plt.xlabel("step"); plt.ylabel("loss")
    plt.title("Curva de perda — SFT QLoRA")
    plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
    plt.savefig("artifacts/fine_tuning/loss_curve.png", dpi=150)
    plt.show()
else:
    print("Sem histórico de perda — o treino não foi executado.")

## 8 e 9. Recarga do adapter e avaliação DEPOIS do treino

In [ ]:
from peft import PeftModel

modelo_base = AutoModelForCausalLM.from_pretrained(MODELO, quantization_config=bnb, device_map="auto")
modelo_tuned = PeftModel.from_pretrained(modelo_base, resultado["adapter_path"])
modelo_tuned.eval()
print("Adapter recarregado de:", resultado["adapter_path"])

for pergunta in PERGUNTAS_SONDA:
    print("PERGUNTA:", pergunta)
    print("ANTES  (base) :", respostas_base[pergunta][:600])
    print("DEPOIS (tuned):", gerar(modelo_tuned, pergunta)[:600])
    print("=" * 100)

In [ ]:
metricas_tuned = evaluate_generation(lambda p: gerar(modelo_tuned, p), sistema="fine_tuned")
print(metricas_tuned.to_dict())

## 10. Comparação base × fine-tuned × fine-tuned + RAG

O terceiro sistema acrescenta o contexto recuperado pelo RAG ao prompt — é a configuração que o
assistente usa em produção.

In [ ]:
from medflow_ai.llm.prompts import build_messages, format_protocol_block
from medflow_ai.graph.tools import get_retriever
from medflow_ai.fine_tuning.evaluate import compare_systems

recuperador = get_retriever()

def com_rag(pergunta: str) -> str:
    chunks = recuperador.retrieve(pergunta)
    mensagens = build_messages(question=pergunta, patient_context="",
                               protocol_context=format_protocol_block(chunks), safety_status="SAFE")
    texto = "\n\n".join(str(m.content) for m in mensagens[1:])
    return gerar(modelo_tuned, texto, max_new_tokens=420)

resultados = compare_systems({
    "base": lambda p: gerar(modelo_base, p),
    "fine_tuned": lambda p: gerar(modelo_tuned, p),
    "fine_tuned_rag": com_rag,
})

import pandas as pd
tabela = pd.DataFrame([
    {k: v for k, v in r.to_dict().items() if k != "exemplos"} for r in resultados
])
tabela

### Como ler esta tabela

Para cada métrica, responda no relatório:

1. **Qual pergunta ela responde?** — por exemplo, `citacao_correta` responde "o sistema aponta o
   documento certo quando afirma algo?";
2. **O que o resultado mostra?**
3. **Qual trade-off apareceu?** — por exemplo, fine-tuning tende a melhorar formato e recusa, mas não
   cria conhecimento factual novo; o ganho de `groundedness` vem do RAG;
4. **É suficiente para o caso de uso?**
5. **Quais limitações permanecem?**

Não conclua superioridade a partir de um único exemplo qualitativo.

## 11. Exportação de artefatos

In [ ]:
import json, pathlib, shutil

destino = pathlib.Path("artifacts/fine_tuning"); destino.mkdir(parents=True, exist_ok=True)
(destino / "comparacao_sistemas.json").write_text(
    json.dumps([r.to_dict() for r in resultados], ensure_ascii=False, indent=2), encoding="utf-8")
(destino / "respostas_antes_depois.json").write_text(json.dumps(
    {p: {"base": respostas_base[p], "fine_tuned": gerar(modelo_tuned, p)} for p in PERGUNTAS_SONDA},
    ensure_ascii=False, indent=2), encoding="utf-8")
print(sorted(p.name for p in destino.iterdir()))

In [ ]:
# Opcional: copiar adapter e métricas para o Google Drive (o adapter NÃO vai para o Git)
# from google.colab import drive
# drive.mount("/content/drive")
# shutil.copytree(resultado["adapter_path"], "/content/drive/MyDrive/medflow-adapter", dirs_exist_ok=True)
# shutil.copy(destino / "comparacao_sistemas.json", "/content/drive/MyDrive/")

## Checklist de encerramento

- [ ] `artifacts/fine_tuning/training_results.json` gerado com `status: ok`
- [ ] `artifacts/fine_tuning/loss_curve.png` salvo
- [ ] `artifacts/fine_tuning/comparacao_sistemas.json` salvo
- [ ] `artifacts/fine_tuning/respostas_antes_depois.json` salvo
- [ ] adapter copiado para o Drive ou Hugging Face (pesos **não** são versionados no Git)
- [ ] tabela comparativa colada em `docs/REPORT.md` com a discussão crítica das 5 perguntas
- [ ] versões das bibliotecas (célula 2) registradas no relatório